# Neuro-Pulse: rPPG-Based Deepfake Detection — ML Training

## Traditional Machine Learning Models

This notebook trains **7 ML classifiers** + a **Voting Ensemble** on the 35-dimensional rPPG feature vectors
extracted from real and deepfake videos.

**Models:**
1. Random Forest
2. XGBoost
3. LightGBM
4. Support Vector Machine (RBF)
5. Gradient Boosting
6. Extra Trees
7. Logistic Regression
8. Soft Voting Ensemble (top-3)

**Pipeline:** Feature extraction → StandardScaler → 3-Fold CV → Pre-Tuned Models → Evaluation

**Dataset:** `/kaggle/input/datasets/likhithvasireddy/deepfake-video-dataset-dip/.../face_dataset_dip/{real_videos, deepfake_videos}`

**References:**
- FakeCatcher (Ciftci et al., TPAMI 2020) — SNR + PSD + MAD + SD + PCC features
- DeepFakesON-Phys (Hernandez-Ortega et al., 2020)
- pyVHR (Boccignone et al., 2022)

## 1. Setup & Imports

In [ ]:
import os
import sys
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from time import time

from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report,
    roc_curve, precision_recall_curve, average_precision_score
)
from sklearn.ensemble import (
    RandomForestClassifier, GradientBoostingClassifier,
    ExtraTreesClassifier, VotingClassifier
)
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression

import xgboost as xgb
import lightgbm as lgb

import joblib

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 12

print(f"NumPy: {np.__version__}")
print(f"XGBoost: {xgb.__version__}")
print(f"LightGBM: {lgb.__version__}")
print("Setup complete.")

## 2. Feature Extraction from Dataset

This cell runs the full video processing pipeline on the Kaggle dataset.
It extracts 35 rPPG features per video and saves them.

**Run this cell ONCE** — it takes time. After that, load from saved files.

In [ ]:
# ─── Configuration ───────────────────────────────────────────────
# Kaggle dataset path
DATASET_ROOT = "/kaggle/input/datasets/likhithvasireddy/deepfake-video-dataset-dip/content/drive/MyDrive/face_dataset_dip"
REAL_DIR = os.path.join(DATASET_ROOT, "real_videos")
FAKE_DIR = os.path.join(DATASET_ROOT, "deepfake_videos")
OUTPUT_DIR = "/kaggle/working/features"
RPPG_METHOD = "GREEN"  # GREEN, CHROM, or POS
MAX_FRAMES_PER_VIDEO = 300  # 10 seconds at 30fps — enough for rPPG, safe on Kaggle

os.makedirs(OUTPUT_DIR, exist_ok=True)

# Verify dataset exists
print(f"Dataset root exists: {os.path.exists(DATASET_ROOT)}")
print(f"Real videos dir exists: {os.path.exists(REAL_DIR)}")
print(f"Fake videos dir exists: {os.path.exists(FAKE_DIR)}")

# If the above shows False, try discovering the actual structure:
if not os.path.exists(REAL_DIR):
    print("\n[WARN] Expected path not found. Scanning dataset root...")
    base = "/kaggle/input"
    for d in os.listdir(base):
        full = os.path.join(base, d)
        if os.path.isdir(full):
            print(f"  /kaggle/input/{d}/")
            for sub in os.listdir(full)[:10]:
                print(f"    {sub}/")
    print("\n[ACTION] Update DATASET_ROOT, REAL_DIR, FAKE_DIR above to match your actual paths.")
else:
    real_files = [f for f in os.listdir(REAL_DIR) if f.endswith(('.mp4', '.avi', '.mov'))]
    print(f"Real videos count: {len(real_files)}")
    fake_files = [f for f in os.listdir(FAKE_DIR) if f.endswith(('.mp4', '.avi', '.mov'))]
    print(f"Fake videos count: {len(fake_files)}")

In [ ]:
# ─── Run Feature Extraction Pipeline ─────────────────────────────
# This processes ALL videos and saves features. Run ONCE.

import glob
import cv2
import mediapipe as mp_module
from scipy.signal import butter, filtfilt, welch, find_peaks, coherence, csd
from scipy.stats import pearsonr, kurtosis, skew
from tqdm import tqdm

# ──────────────────────────────────────────────────────────────────
# Inline all necessary functions so this notebook is self-contained
# ──────────────────────────────────────────────────────────────────

# ROI landmark indices (MediaPipe Face Mesh)
ROI_FOREHEAD = [10, 338, 297, 332, 284, 251, 389, 356, 454, 323, 361,
                288, 397, 365, 379, 378, 400, 377, 152, 148, 176, 149,
                150, 136, 172, 58, 132, 93, 234, 127, 162, 21, 54, 103, 67, 109]
ROI_LEFT_CHEEK = [187, 123, 116, 117, 118, 119, 120, 121, 128, 245, 193, 55, 65, 52, 53]
ROI_RIGHT_CHEEK = [411, 352, 345, 346, 347, 348, 349, 350, 357, 465, 417, 285, 295, 282, 283]

BANDPASS_LOW = 0.7
BANDPASS_HIGH = 3.5
BANDPASS_ORDER = 4

FEATURE_NAMES = [
    "fh_snr", "fh_spectral_purity", "fh_peak_prominence",
    "fh_dominant_freq", "fh_harmonic_ratio", "fh_spectral_entropy",
    "fh_spectral_centroid",
    "fh_mad", "fh_std", "fh_zcr", "fh_kurtosis", "fh_skewness",
    "lc_snr", "lc_spectral_purity", "lc_peak_prominence",
    "lc_dominant_freq", "lc_harmonic_ratio", "lc_spectral_entropy",
    "lc_spectral_centroid",
    "lc_mad", "lc_std", "lc_zcr", "lc_kurtosis", "lc_skewness",
    "corr_fh_lc", "corr_fh_rc", "corr_lc_rc",
    "coherence_fh_lc", "coherence_fh_rc", "coherence_lc_rc",
    "phase_diff_fh_lc", "phase_diff_fh_rc",
    "bpm_estimate", "signal_stationarity", "bpm_consistency",
]


def bandpass_filter(signal, fs, lowcut=0.7, highcut=3.5, order=4):
    nyq = 0.5 * fs
    low = max(lowcut / nyq, 0.001)
    high = min(highcut / nyq, 0.999)
    if low >= high:
        return signal
    b, a = butter(order, [low, high], btype='band')
    if len(signal) < 3 * max(len(a), len(b)):
        return signal
    return filtfilt(b, a, signal)


def detrend_signal(signal, lam=300):
    n = len(signal)
    if n < 5:
        return signal
    I = np.eye(n)
    D2 = np.zeros((n - 2, n))
    for i in range(n - 2):
        D2[i, i] = 1; D2[i, i+1] = -2; D2[i, i+2] = 1
    return signal - np.linalg.solve(I + lam**2 * D2.T @ D2, signal)


def extract_green(rgb_mean):
    return rgb_mean[:, 1].copy()


def extract_chrom(rgb_mean, fs=30):
    r = rgb_mean[:, 0] / (np.mean(rgb_mean[:, 0]) + 1e-8)
    g = rgb_mean[:, 1] / (np.mean(rgb_mean[:, 1]) + 1e-8)
    b = rgb_mean[:, 2] / (np.mean(rgb_mean[:, 2]) + 1e-8)
    xs = 3*r - 2*g
    ys = 1.5*r + g - 1.5*b
    win = max(int(1.6*fs), 2)
    bvp = np.zeros(len(r))
    for i in range(len(r)):
        s, e = max(0, i-win//2), min(len(r), i+win//2)
        alpha = np.std(xs[s:e]) / (np.std(ys[s:e]) + 1e-8)
        bvp[i] = xs[i] - alpha * ys[i]
    return bvp


def extract_pos(rgb_mean, fs=30):
    n = rgb_mean.shape[0]
    win = max(int(1.6*fs), 2)
    bvp = np.zeros(n)
    for i in range(n):
        s = max(0, i-win+1)
        if i-s < 2: continue
        w = rgb_mean[s:i+1]
        m = np.mean(w, axis=0); m[m<1e-8] = 1e-8
        cn = w / m
        s1 = cn[:,1]-cn[:,2]; s2 = cn[:,1]+cn[:,2]-2*cn[:,0]
        alpha = np.std(s1)/(np.std(s2)+1e-8)
        bvp[i] = (s1+alpha*s2)[-1]
    return bvp


RPPG_METHODS = {"GREEN": extract_green, "CHROM": extract_chrom, "POS": extract_pos}


def compute_welch_psd(bvp, fs, nperseg=256, noverlap=128, nfft=1024):
    nperseg = min(nperseg, len(bvp))
    noverlap = min(noverlap, nperseg - 1)
    return welch(bvp, fs=fs, nperseg=nperseg, noverlap=noverlap, nfft=nfft)


def get_roi_mean_rgb(frame, landmarks, roi_indices, h, w):
    pts = []
    for idx in roi_indices:
        lm = landmarks.landmark[idx]
        x = max(0, min(int(lm.x * w), w-1))
        y = max(0, min(int(lm.y * h), h-1))
        pts.append((x, y))
    if len(pts) < 3: return None
    pts = np.array(pts, dtype=np.int32)
    x0, y0 = np.min(pts, axis=0); x1, y1 = np.max(pts, axis=0)
    x0, y0, x1, y1 = max(0,x0), max(0,y0), min(w,x1), min(h,y1)
    if x1<=x0 or y1<=y0: return None
    roi = frame[y0:y1, x0:x1]
    if roi.size==0: return None
    return np.mean(roi.reshape(-1,3), axis=0)[::-1].astype(np.float64)


def _extract_35_features(bvp_fh, bvp_lc, bvp_rc, fs):
    """Extract the full 35-feature vector."""
    fl = BANDPASS_LOW
    fh = BANDPASS_HIGH
    
    def _roi_features(bvp):
        if len(bvp) < 10:
            return [0.0]*12
        freqs, psd = compute_welch_psd(bvp, fs)
        mask = (freqs >= fl) & (freqs <= fh)
        if not np.any(mask):
            return [0.0]*12
        mp_ = psd[mask]; mf = freqs[mask]
        pi = np.argmax(mp_); pf = mf[pi]
        # SNR
        sm = np.abs(mf-pf)<=0.2; nm = ~sm
        sp_ = np.sum(mp_[sm]); np_ = np.sum(mp_[nm])
        snr = 10*np.log10(sp_/(np_+1e-10)) if np_>1e-10 else 40.0
        # Spectral purity
        spur = sp_/(np.sum(mp_)+1e-10)
        # Peak prominence
        peaks, props = find_peaks(mp_, prominence=0)
        pp = float(np.max(props['prominences'])) if len(peaks)>0 else 0.0
        # Harmonic ratio
        hf = 2*pf
        hm = (freqs>=hf-0.15)&(freqs<=hf+0.15)
        hr = float(np.max(psd[hm])/mp_[pi]) if np.any(hm) and mp_[pi]>1e-10 else 0.0
        # Spectral entropy
        pn = mp_/(np.sum(mp_)+1e-10); pn = pn[pn>0]
        se = float(-np.sum(pn*np.log2(pn+1e-10)))
        # Spectral centroid
        sc = float(np.sum(mf*mp_)/(np.sum(mp_)+1e-10))
        # Temporal
        mad = float(np.mean(np.abs(bvp-np.mean(bvp))))
        std = float(np.std(bvp))
        zcr = float(np.sum(np.abs(np.diff(np.sign(bvp)))>0)/(len(bvp)-1)) if len(bvp)>1 else 0.0
        kurt = float(kurtosis(bvp)) if len(bvp)>3 else 0.0
        skewn = float(skew(bvp)) if len(bvp)>3 else 0.0
        return [snr, spur, pp, float(pf), hr, se, sc, mad, std, zcr, kurt, skewn]
    
    fh_f = _roi_features(bvp_fh)
    lc_f = _roi_features(bvp_lc)
    
    # Cross-ROI
    def _corr(a, b):
        ml = min(len(a), len(b))
        if ml<3: return 0.0
        c, _ = pearsonr(a[:ml], b[:ml])
        return 0.0 if np.isnan(c) else float(c)
    
    def _coh(a, b):
        ns = min(256, min(len(a), len(b)))
        if ns<4: return 0.0
        f, c = coherence(a, b, fs=fs, nperseg=ns)
        m = (f>=fl)&(f<=fh)
        return float(np.mean(c[m])) if np.any(m) else 0.0
    
    def _phase(a, b):
        ns = min(256, min(len(a), len(b)))
        if ns<4: return 0.0
        f, pxy = csd(a, b, fs=fs, nperseg=ns)
        m = (f>=fl)&(f<=fh)
        if not np.any(m): return 0.0
        return float(np.abs(np.angle(pxy[m][np.argmax(np.abs(pxy[m]))])))
    
    c_fl = _corr(bvp_fh, bvp_lc)
    c_fr = _corr(bvp_fh, bvp_rc)
    c_lr = _corr(bvp_lc, bvp_rc)
    coh_fl = _coh(bvp_fh, bvp_lc)
    coh_fr = _coh(bvp_fh, bvp_rc)
    coh_lr = _coh(bvp_lc, bvp_rc)
    ph_fl = _phase(bvp_fh, bvp_lc)
    ph_fr = _phase(bvp_fh, bvp_rc)
    
    # BPM
    def _bpm(bvp):
        f, p = compute_welch_psd(bvp, fs)
        m = (f>=fl)&(f<=fh)
        if not np.any(m): return 0.0
        return float(f[m][np.argmax(p[m])]*60)
    
    bpm = _bpm(bvp_fh)
    bpm_lc = _bpm(bvp_lc)
    bpm_rc = _bpm(bvp_rc)
    
    # Stationarity
    segs = np.array_split(bvp_fh, 4) if len(bvp_fh)>=8 else [bvp_fh]
    vs = [np.var(s) for s in segs]
    mv = np.mean(vs)
    stat = max(0.0, 1.0 - np.std(vs)/(mv+1e-10)) if mv>1e-10 else 1.0
    
    # BPM consistency
    bpms = [bpm, bpm_lc, bpm_rc]
    bm = np.mean(bpms)
    bcon = max(0.0, min(1.0, 1.0 - np.std(bpms)/(bm+1e-8)))
    
    feat = np.array(fh_f + lc_f + [
        c_fl, c_fr, c_lr, coh_fl, coh_fr, coh_lr, ph_fl, ph_fr,
        bpm, stat, bcon
    ], dtype=np.float64)
    return np.nan_to_num(feat, nan=0.0, posinf=40.0, neginf=-40.0)


# ──────────────────────────────────────────────────────────────────
# FIX 3: Initialize FaceMesh ONCE — reused for all videos
# refine_landmarks=False saves ~30% per frame (we don't use eye/lip refinement)
# ──────────────────────────────────────────────────────────────────
_FACE_MESH = mp_module.solutions.face_mesh.FaceMesh(
    static_image_mode=False,
    max_num_faces=1,
    refine_landmarks=False,
    min_detection_confidence=0.5,
    min_tracking_confidence=0.5,
)


def process_video(video_path, method="GREEN", max_frames=300):
    """Extract 35 rPPG features from a single video. Uses shared FaceMesh instance."""
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        return None
    fps = cap.get(cv2.CAP_PROP_FPS)
    if fps <= 0 or fps > 120: fps = 30.0
    
    rgb_fh, rgb_lc, rgb_rc = [], [], []
    fc = 0
    while fc < max_frames:
        ret, frame = cap.read()
        if not ret: break
        fc += 1
        h, w = frame.shape[:2]
        res = _FACE_MESH.process(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
        if res.multi_face_landmarks:
            lm = res.multi_face_landmarks[0]
            f1 = get_roi_mean_rgb(frame, lm, ROI_FOREHEAD, h, w)
            f2 = get_roi_mean_rgb(frame, lm, ROI_LEFT_CHEEK, h, w)
            f3 = get_roi_mean_rgb(frame, lm, ROI_RIGHT_CHEEK, h, w)
            if f1 is not None and f2 is not None and f3 is not None:
                rgb_fh.append(f1); rgb_lc.append(f2); rgb_rc.append(f3)
    cap.release()
    # Do NOT close _FACE_MESH — it's reused across all videos
    
    if len(rgb_fh) < int(2*fps): return None
    
    efn = RPPG_METHODS.get(method, extract_green)
    bvp_fh = bandpass_filter(detrend_signal(efn(np.array(rgb_fh))), fps)
    bvp_lc = bandpass_filter(detrend_signal(efn(np.array(rgb_lc))), fps)
    bvp_rc = bandpass_filter(detrend_signal(efn(np.array(rgb_rc))), fps)
    
    return _extract_35_features(bvp_fh, bvp_lc, bvp_rc, fps)


print("Feature extraction functions defined.")
print(f"FaceMesh initialized ONCE (refine_landmarks=False).")
print(f"Max frames per video: {MAX_FRAMES_PER_VIDEO}")

In [ ]:
# ─── Process all videos and save features ────────────────────────
# Skip this cell if features are already extracted

FEATURES_FILE = os.path.join(OUTPUT_DIR, "features.npy")
LABELS_FILE = os.path.join(OUTPUT_DIR, "labels.npy")

if os.path.exists(FEATURES_FILE) and os.path.exists(LABELS_FILE):
    print("Features already extracted. Loading from files...")
    X = np.load(FEATURES_FILE)
    y = np.load(LABELS_FILE)
    print(f"Loaded: X={X.shape}, y={y.shape}")
    print(f"Real: {np.sum(y==0)}, Fake: {np.sum(y==1)}")
else:
    video_exts = ('*.mp4', '*.avi', '*.mov', '*.mkv', '*.webm')
    
    real_vids = []
    fake_vids = []
    for ext in video_exts:
        real_vids.extend(glob.glob(os.path.join(REAL_DIR, ext)))
        real_vids.extend(glob.glob(os.path.join(REAL_DIR, ext.upper())))
        fake_vids.extend(glob.glob(os.path.join(FAKE_DIR, ext)))
        fake_vids.extend(glob.glob(os.path.join(FAKE_DIR, ext.upper())))
    
    print(f"Found {len(real_vids)} real, {len(fake_vids)} fake videos")
    
    all_X, all_y = [], []
    
    print("\nProcessing REAL videos...")
    for vp in tqdm(real_vids, desc="Real"):
        feat = process_video(vp, RPPG_METHOD, MAX_FRAMES_PER_VIDEO)
        if feat is not None:
            all_X.append(feat); all_y.append(0)
    
    print("\nProcessing FAKE videos...")
    for vp in tqdm(fake_vids, desc="Fake"):
        feat = process_video(vp, RPPG_METHOD, MAX_FRAMES_PER_VIDEO)
        if feat is not None:
            all_X.append(feat); all_y.append(1)
    
    X = np.array(all_X, dtype=np.float64)
    y = np.array(all_y, dtype=np.int64)
    
    np.save(FEATURES_FILE, X)
    np.save(LABELS_FILE, y)
    
    # Also save CSV
    df = pd.DataFrame(X, columns=FEATURE_NAMES)
    df['label'] = y
    df.to_csv(os.path.join(OUTPUT_DIR, "features.csv"), index=False)
    
    print(f"\nDone! X={X.shape}, y={y.shape}")
    print(f"Real: {np.sum(y==0)}, Fake: {np.sum(y==1)}")
    print(f"Saved to {OUTPUT_DIR}")

## 3. Exploratory Data Analysis

In [ ]:
# ─── Dataset Summary ─────────────────────────────────────────────
df = pd.DataFrame(X, columns=FEATURE_NAMES)
df['label'] = y

print("=" * 60)
print("DATASET SUMMARY")
print("=" * 60)
print(f"Total samples: {len(df)}")
print(f"Features: {X.shape[1]}")
print(f"\nClass distribution:")
print(df['label'].value_counts().rename({0: 'Real', 1: 'Fake'}))
print(f"\nClass ratio: {np.sum(y==0)/np.sum(y==1):.2f} (Real/Fake)")
print(f"\nFeature statistics:")
print(df.describe().T[['mean', 'std', 'min', 'max']].to_string())

In [ ]:
# ─── Feature Distributions: Real vs Fake ─────────────────────────
fig, axes = plt.subplots(6, 6, figsize=(24, 20))
axes = axes.flatten()

for i, fname in enumerate(FEATURE_NAMES):
    ax = axes[i]
    ax.hist(X[y==0, i], bins=30, alpha=0.6, label='Real', color='green', density=True)
    ax.hist(X[y==1, i], bins=30, alpha=0.6, label='Fake', color='red', density=True)
    ax.set_title(fname, fontsize=9)
    ax.legend(fontsize=7)

# Hide extra subplot
axes[35].set_visible(False)

plt.suptitle('Feature Distributions: Real vs Deepfake', fontsize=16, y=1.01)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'feature_distributions.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ─── Correlation Matrix ──────────────────────────────────────────
fig, ax = plt.subplots(figsize=(16, 14))
corr = df[FEATURE_NAMES].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=False, cmap='coolwarm', center=0,
            square=True, linewidths=0.5, ax=ax)
ax.set_title('Feature Correlation Matrix', fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'correlation_matrix.png'), dpi=150, bbox_inches='tight')
plt.show()

## 4. Data Preprocessing

In [ ]:
# ─── Train/Test Split + Scaling ──────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Train: {X_train_scaled.shape} | Test: {X_test_scaled.shape}")
print(f"Train class dist: Real={np.sum(y_train==0)}, Fake={np.sum(y_train==1)}")
print(f"Test  class dist: Real={np.sum(y_test==0)}, Fake={np.sum(y_test==1)}")

# Save scaler for inference
joblib.dump(scaler, os.path.join(OUTPUT_DIR, 'scaler.joblib'))
print("Scaler saved.")

## 5. Model Definitions (Pre-Tuned Hyperparameters — No Grid Search)

In [ ]:
# ─── All models with FIXED pre-tuned hyperparameters ─────────────
# No GridSearchCV = no crash, finishes in minutes.

SEED = 42

models = {
    'RandomForest': RandomForestClassifier(
        n_estimators=300, max_depth=20, min_samples_split=5,
        min_samples_leaf=2, max_features='sqrt',
        random_state=SEED, n_jobs=-1,
    ),
    'XGBoost': xgb.XGBClassifier(
        n_estimators=300, max_depth=5, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8,
        reg_alpha=0.1, reg_lambda=2.0,
        eval_metric='logloss', random_state=SEED, n_jobs=-1,
    ),
    'LightGBM': lgb.LGBMClassifier(
        n_estimators=300, max_depth=10, learning_rate=0.05,
        num_leaves=63, subsample=0.8, colsample_bytree=0.8,
        reg_alpha=0.1, reg_lambda=1.0,
        random_state=SEED, n_jobs=-1, verbose=-1,
    ),
    'SVM_RBF': SVC(
        C=10.0, gamma='scale', kernel='rbf',
        probability=True, random_state=SEED,
    ),
    'GradientBoosting': GradientBoostingClassifier(
        n_estimators=300, max_depth=5, learning_rate=0.05,
        subsample=0.8, min_samples_split=5,
        random_state=SEED,
    ),
    'ExtraTrees': ExtraTreesClassifier(
        n_estimators=300, max_depth=20, min_samples_split=5,
        max_features='sqrt', random_state=SEED, n_jobs=-1,
    ),
    'LogisticRegression': LogisticRegression(
        C=1.0, penalty='l2', solver='saga',
        max_iter=2000, random_state=SEED,
    ),
}

print(f"Defined {len(models)} models (pre-tuned, no grid search).")
for name, model in models.items():
    print(f"  {name}")

## 6. Train All Models (Direct Fit + 3-Fold CV)

In [ ]:
# ─── Train each model: fit on train, evaluate on test ────────────
# Also run quick 3-fold CV for reporting in the paper.

from sklearn.model_selection import cross_val_score

results = {}
best_models = {}

for name, model in models.items():
    print(f"\n{'='*60}")
    print(f"Training: {name}")
    print(f"{'='*60}")
    
    t0 = time()
    
    # 3-fold CV on training set (for paper reporting)
    cv_scores = cross_val_score(model, X_train_scaled, y_train,
                                cv=3, scoring='roc_auc', n_jobs=-1)
    
    # Fit on full training set
    model.fit(X_train_scaled, y_train)
    
    elapsed = time() - t0
    
    # Evaluate on test set
    y_pred = model.predict(X_test_scaled)
    y_prob = model.predict_proba(X_test_scaled)[:, 1]
    
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, zero_division=0)
    rec = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)
    auc = roc_auc_score(y_test, y_prob)
    
    results[name] = {
        'accuracy': acc, 'precision': prec, 'recall': rec,
        'f1': f1, 'auc': auc,
        'cv_auc_mean': cv_scores.mean(),
        'cv_auc_std': cv_scores.std(),
        'time': elapsed,
    }
    best_models[name] = model
    
    print(f"  CV AUC: {cv_scores.mean():.4f} +/- {cv_scores.std():.4f}")
    print(f"  Test Accuracy: {acc:.4f}")
    print(f"  Test AUC: {auc:.4f}")
    print(f"  Test F1: {f1:.4f}")
    print(f"  Time: {elapsed:.1f}s")
    
    # Save model
    joblib.dump(model, os.path.join(OUTPUT_DIR, f'{name}_best.joblib'))

print("\n" + "="*60)
print("All models trained and saved!")
print("="*60)

## 7. Ensemble Model (Soft Voting — Top 3 by AUC)

In [ ]:
# ─── Build ensemble from top-3 models by AUC ────────────────────

sorted_models = sorted(results.items(), key=lambda x: x[1]['auc'], reverse=True)
top3_names = [name for name, _ in sorted_models[:3]]

print(f"Top 3 models by AUC: {top3_names}")
for name in top3_names:
    print(f"  {name}: AUC={results[name]['auc']:.4f}")

# Create voting ensemble
ensemble_estimators = [(name, best_models[name]) for name in top3_names]
ensemble = VotingClassifier(
    estimators=ensemble_estimators,
    voting='soft',
    n_jobs=-1,
)

# Train ensemble (re-fits on training data)
ensemble.fit(X_train_scaled, y_train)

# Evaluate
y_pred_ens = ensemble.predict(X_test_scaled)
y_prob_ens = ensemble.predict_proba(X_test_scaled)[:, 1]

ens_acc = accuracy_score(y_test, y_pred_ens)
ens_auc = roc_auc_score(y_test, y_prob_ens)
ens_f1 = f1_score(y_test, y_pred_ens)
ens_prec = precision_score(y_test, y_pred_ens)
ens_rec = recall_score(y_test, y_pred_ens)

results['Ensemble_Top3'] = {
    'accuracy': ens_acc, 'precision': ens_prec, 'recall': ens_rec,
    'f1': ens_f1, 'auc': ens_auc,
    'cv_auc_mean': None,  # No CV for ensemble
    'cv_auc_std': None,
    'time': None,
}
best_models['Ensemble_Top3'] = ensemble

print(f"\nEnsemble Accuracy: {ens_acc:.4f}")
print(f"Ensemble AUC: {ens_auc:.4f}")
print(f"Ensemble F1: {ens_f1:.4f}")
print(f"Members: {top3_names}")

joblib.dump(ensemble, os.path.join(OUTPUT_DIR, 'Ensemble_Top3_best.joblib'))

## 8. Results Comparison Table

In [ ]:
# ─── Results Table ───────────────────────────────────────────────

results_df = pd.DataFrame(results).T
results_df = results_df[['accuracy', 'precision', 'recall', 'f1', 'auc', 'cv_auc_mean', 'cv_auc_std', 'time']]
results_df = results_df.sort_values('auc', ascending=False)

display_df = results_df.copy()
for col in ['accuracy', 'precision', 'recall', 'f1', 'auc']:
    display_df[col] = display_df[col].apply(lambda x: f"{x:.4f}" if pd.notna(x) and x is not None else "-")
for col in ['cv_auc_mean', 'cv_auc_std']:
    display_df[col] = display_df[col].apply(lambda x: f"{x:.4f}" if pd.notna(x) and x is not None else "-")
display_df['time'] = display_df['time'].apply(lambda x: f"{x:.1f}s" if pd.notna(x) and x is not None else "-")

print("\n" + "="*80)
print("MODEL COMPARISON — SORTED BY AUC")
print("="*80)
print(display_df.to_string())

# Save results
results_df.to_csv(os.path.join(OUTPUT_DIR, 'ml_results.csv'))
print(f"\nResults saved to {OUTPUT_DIR}/ml_results.csv")

## 9. Visualization: ROC Curves, Confusion Matrices, Feature Importance

In [ ]:
# ─── ROC Curves for All Models ───────────────────────────────────

fig, ax = plt.subplots(figsize=(10, 8))

for name, model in best_models.items():
    y_prob = model.predict_proba(X_test_scaled)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    auc_val = roc_auc_score(y_test, y_prob)
    lw = 3 if 'Ensemble' in name else 1.5
    ax.plot(fpr, tpr, label=f"{name} (AUC={auc_val:.4f})", linewidth=lw)

ax.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random')
ax.set_xlabel('False Positive Rate', fontsize=13)
ax.set_ylabel('True Positive Rate', fontsize=13)
ax.set_title('ROC Curves — All Models', fontsize=15)
ax.legend(loc='lower right', fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'roc_curves.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ─── Confusion Matrices ──────────────────────────────────────────

n_models = len(best_models)
cols = 4
rows = (n_models + cols - 1) // cols

fig, axes = plt.subplots(rows, cols, figsize=(5*cols, 4*rows))
axes = axes.flatten()

for i, (name, model) in enumerate(best_models.items()):
    y_pred = model.predict(X_test_scaled)
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[i],
                xticklabels=['Real', 'Fake'], yticklabels=['Real', 'Fake'])
    axes[i].set_title(f'{name}\nAcc={accuracy_score(y_test, y_pred):.3f}', fontsize=10)
    axes[i].set_xlabel('Predicted')
    axes[i].set_ylabel('Actual')

for j in range(i+1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Confusion Matrices — All Models', fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'confusion_matrices.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ─── Feature Importance (from tree-based models) ─────────────────

tree_models = ['RandomForest', 'XGBoost', 'LightGBM', 'ExtraTrees', 'GradientBoosting']

fig, axes = plt.subplots(1, len(tree_models), figsize=(6*len(tree_models), 10))

for i, name in enumerate(tree_models):
    model = best_models[name]
    importance = model.feature_importances_
    sorted_idx = np.argsort(importance)
    
    axes[i].barh(range(len(sorted_idx)), importance[sorted_idx], align='center')
    axes[i].set_yticks(range(len(sorted_idx)))
    axes[i].set_yticklabels([FEATURE_NAMES[j] for j in sorted_idx], fontsize=7)
    axes[i].set_title(f'{name}', fontsize=11)
    axes[i].set_xlabel('Importance')

plt.suptitle('Feature Importance — Tree-Based Models', fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'feature_importance.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ─── Precision-Recall Curves ─────────────────────────────────────

fig, ax = plt.subplots(figsize=(10, 8))

for name, model in best_models.items():
    y_prob = model.predict_proba(X_test_scaled)[:, 1]
    prec_vals, rec_vals, _ = precision_recall_curve(y_test, y_prob)
    ap = average_precision_score(y_test, y_prob)
    lw = 3 if 'Ensemble' in name else 1.5
    ax.plot(rec_vals, prec_vals, label=f"{name} (AP={ap:.4f})", linewidth=lw)

ax.set_xlabel('Recall', fontsize=13)
ax.set_ylabel('Precision', fontsize=13)
ax.set_title('Precision-Recall Curves — All Models', fontsize=15)
ax.legend(loc='lower left', fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'precision_recall_curves.png'), dpi=150, bbox_inches='tight')
plt.show()

## 10. Best Model Classification Report

In [ ]:
# ─── Detailed report for the best model ──────────────────────────

best_name = results_df.index[0]
best_model = best_models[best_name]

y_pred_best = best_model.predict(X_test_scaled)
y_prob_best = best_model.predict_proba(X_test_scaled)[:, 1]

print(f"{'='*60}")
print(f"BEST MODEL: {best_name}")
print(f"{'='*60}")
print(f"\nTest AUC: {roc_auc_score(y_test, y_prob_best):.4f}")

# CV AUC — only available for individual models, not Ensemble
cv_mean = results[best_name].get('cv_auc_mean')
cv_std = results[best_name].get('cv_auc_std')
if cv_mean is not None and cv_std is not None:
    print(f"CV AUC: {cv_mean:.4f} +/- {cv_std:.4f}")
else:
    print(f"CV AUC: N/A (ensemble of top-3 models)")

print(f"\nClassification Report:")
print(classification_report(y_test, y_pred_best, target_names=['Real', 'Deepfake']))

## 11. Save Everything for Deep Learning Notebook

In [ ]:
# ─── Save all artifacts ──────────────────────────────────────────

# Save train/test split for consistency with DL notebook
np.save(os.path.join(OUTPUT_DIR, 'X_train.npy'), X_train)
np.save(os.path.join(OUTPUT_DIR, 'X_test.npy'), X_test)
np.save(os.path.join(OUTPUT_DIR, 'y_train.npy'), y_train)
np.save(os.path.join(OUTPUT_DIR, 'y_test.npy'), y_test)
np.save(os.path.join(OUTPUT_DIR, 'X_train_scaled.npy'), X_train_scaled)
np.save(os.path.join(OUTPUT_DIR, 'X_test_scaled.npy'), X_test_scaled)

print("Saved artifacts:")
for f in os.listdir(OUTPUT_DIR):
    fpath = os.path.join(OUTPUT_DIR, f)
    size_mb = os.path.getsize(fpath) / 1024 / 1024
    print(f"  {f}: {size_mb:.2f} MB")

print(f"\nTotal models saved: {len(best_models)}")
print("ML training complete. Proceed to Deep Learning notebook for DL models.")